# CNN Food Classifier — Local Training Walkthrough

This notebook walks through training a **Convolutional Neural Network** (CNN) to classify food images into **101 categories** using **transfer learning** with MobileNetV2 pretrained on ImageNet.

It covers:
- What CNNs are and why they work for image classification
- How MobileNetV2 transfer learning works
- Loading and preprocessing Food-101
- Building and training the model
- Evaluating results with visualisations
- How to fine-tune by unfreezing base layers

**Prerequisites:** Python 3.11+, install dependencies with `pip install -r requirements.txt`

## 1. What is a CNN?

A **Convolutional Neural Network** is a type of neural network designed for image data. Unlike a fully-connected network that treats every pixel independently, a CNN uses **convolutional filters** — small learnable kernels that slide across the image and detect local patterns like edges, textures, and shapes.

The key operations:
- **Conv2D**: applies filters to detect features at each spatial location
- **MaxPooling**: down-samples the feature map, reducing spatial dimensions
- **ReLU activation**: introduces non-linearity so the network can learn complex patterns
- **Dense (fully-connected) layers**: map extracted features to class probabilities

Early layers detect low-level features (edges, gradients). Deeper layers detect higher-level patterns (textures, object parts, whole objects).

## 2. How MobileNetV2 Transfer Learning Works

Training a CNN from scratch on 101 food categories would require hundreds of thousands of images and hours of compute. **Transfer learning** sidesteps this by starting with a model already trained on a large dataset.

**MobileNetV2** was trained on **ImageNet** — 1.2 million images, 1,000 classes. Its convolutional base has already learned to recognise general visual features: textures, edges, colours, shapes. These features are broadly useful for food classification even though ImageNet doesn't contain Food-101 images.

### Stage 1 — Feature Extraction (what we do here)
1. Load MobileNetV2 with `include_top=False` — drop its original classification head
2. Freeze all base layers: `base_model.trainable = False`
3. Add a new classification head: GlobalAveragePooling → Dense(256) → Dropout → Dense(101, softmax)
4. Train only the new head — the base stays fixed as a feature extractor

This converges quickly (10 epochs) and achieves ~70% validation accuracy with minimal compute.

### Stage 2 — Fine-tuning (optional, see section at the end)
Unfreeze the top layers of the base model and retrain with a very low learning rate. This pushes accuracy higher at the cost of longer training.

In [ ]:
import tensorflow as tf          # the main deep learning framework
import tensorflow_datasets as tfds  # handles downloading and loading datasets
import numpy as np                  # numerical operations on arrays
import matplotlib.pyplot as plt     # plotting training curves and predictions
import seaborn as sns               # nicer-looking heatmaps for the confusion matrix
from pathlib import Path            # cleaner file path handling than raw strings
from sklearn.metrics import confusion_matrix  # evaluates where the model makes mistakes
import time                         # used to measure total training duration

# Use 16-bit floats instead of 32-bit where possible — roughly 30% faster on
# M1 Metal and modern GPUs because more data fits in GPU memory per operation.
# Accuracy is unaffected. The final output layer is kept at float32 (see
# build_model) to avoid precision errors in the softmax probability calculation.
tf.keras.mixed_precision.set_global_policy('mixed_float16')

# Print this upfront so we can confirm Metal GPU acceleration is active
# before committing to a potentially long training run
print('TensorFlow:', tf.__version__)
print('GPU devices:', tf.config.list_physical_devices('GPU'))
print('Mixed precision policy:', tf.keras.mixed_precision.global_policy().name)

PLOTS_DIR = Path('plots')
CHECKPOINT_DIR = Path('checkpoints')
PLOTS_DIR.mkdir(exist_ok=True)
CHECKPOINT_DIR.mkdir(exist_ok=True)

IMG_SIZE = 224      # MobileNetV2 was designed for 224x224 images — all inputs must match
BATCH_SIZE = 32     # how many images to process at once before updating the model weights
EPOCHS = 10         # how many times to loop through the full training set
NUM_CLASSES = 101   # one output neuron per food category
MODEL_PATH = 'food_classifier.h5'

## 3. Dataset — Food-101

**Food-101** contains 101,000 images across 101 food categories:
- 750 training images per class (75,750 total)
- 250 test images per class (25,250 total)

Images are RGB JPEG files of varying sizes. We resize everything to 224×224 to match MobileNetV2's expected input.

We load via `tensorflow_datasets` which handles downloading, caching, and providing a `tf.data.Dataset` pipeline.

In [ ]:
# tensorflow_datasets handles downloading, verifying, and caching Food-101
# automatically — no manual unzipping or folder setup required.
# as_supervised=True returns (image, label) tuples instead of raw dicts,
# which is the format the training pipeline expects.
print('Loading Food-101...')
(train_ds_raw, val_ds_raw), info = tfds.load(
    'food101',
    split=['train', 'validation'],
    as_supervised=True,
    with_info=True   # also returns metadata: class names, image counts, etc.
)

class_names = info.features['label'].names  # list of 101 food category strings
print(f'Train: {info.splits["train"].num_examples} images')
print(f'Val  : {info.splits["validation"].num_examples} images')
print(f'Classes: {len(class_names)}')
print('First 10 classes:', class_names[:10])

## 4. Preprocessing & Data Augmentation

**Preprocessing** (applied to both train and val):
- Resize to 224×224
- Normalise pixel values from [0, 255] to [0, 1]

**Augmentation** (training only) artificially expands the training set and reduces overfitting:
- `RandomFlip('horizontal')` — mirrors images left-right
- `RandomRotation(0.1)` — rotates ±10%
- `RandomZoom(0.1)` — zooms in/out by ±10%

The `tf.data` pipeline uses `.shuffle().prefetch(AUTOTUNE)` to keep the GPU fed. `.cache()` is intentionally omitted — caching the full Food-101 dataset at 224×224 float32 requires ~14GB of RAM, which would exhaust the available memory on a 16GB machine.

In [ ]:
def preprocess(image, label):
    # Food-101 images come in various sizes — resize everything to the fixed
    # 224x224 that MobileNetV2 expects so all inputs have the same shape
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    # Scale pixel values from 0–255 integers to 0.0–1.0 floats.
    # Neural networks converge faster and more stably when inputs are small
    # numbers rather than large integers.
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

# Augmentation applies random transforms to training images each epoch,
# so the model sees slightly different versions of each photo every time.
# This discourages memorisation and improves generalisation to new images.
# These transforms are only applied during training — val images are untouched.
augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),   # mirror left-right at random
    tf.keras.layers.RandomRotation(0.1),        # rotate up to ~36 degrees either way
    tf.keras.layers.RandomZoom(0.1),            # zoom in or out by up to 10%
])

# Build the tf.data pipelines.
# Each step is chained so data flows from disk → CPU → GPU without bottlenecks.
# .shuffle(1000) randomises order so the model doesn't learn the sequence of images.
# .prefetch(AUTOTUNE) prepares the next batch on CPU while the GPU trains on the current one.
# .cache() is intentionally omitted — loading all 75k images at 224x224 into RAM
# would require ~14GB, which risks crashing on a 16GB machine.
train_dataset = (
    train_ds_raw
    .map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .shuffle(1000)
    .batch(BATCH_SIZE)
    .map(lambda x, y: (augmentation(x, training=True), y),
         num_parallel_calls=tf.data.AUTOTUNE)
    .prefetch(tf.data.AUTOTUNE)
)

# Validation pipeline — no augmentation, just resize and normalise
val_dataset = (
    val_ds_raw
    .map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

print('Pipeline ready.')

## 5. Model Architecture

```
Input (224 × 224 × 3)
  └─ MobileNetV2 base (pretrained ImageNet, weights frozen)
       └─ GlobalAveragePooling2D   ← pools spatial dims to a 1280-d vector
            └─ Dense(256, ReLU)
                 └─ Dropout(0.3)   ← regularisation
                      └─ Dense(101, Softmax)
                           └─ Output: 101 class probabilities
```

**GlobalAveragePooling2D** averages each feature map across spatial dimensions, reducing the MobileNetV2 output (7×7×1280) to a 1280-dimensional vector. This is more parameter-efficient and less prone to overfitting than Flatten (which would produce 62,720 values).

In [ ]:
def build_model():
    # Load MobileNetV2 pretrained on ImageNet — it has already learned to detect
    # edges, textures, and shapes from 1.2 million images across 1,000 categories.
    # include_top=False removes its original 1,000-class output layer so we can
    # attach our own 101-class head for food classification.
    base_model = tf.keras.applications.MobileNetV2(
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        include_top=False,
        weights='imagenet'
    )
    # Freeze the base — we want to reuse its learned features, not overwrite them.
    # Only the new classification head will be trained.
    base_model.trainable = False

    inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    # training=False keeps BatchNorm layers in inference mode while the base is frozen
    x = base_model(inputs, training=False)

    # MobileNetV2 outputs a 7x7x1280 feature map. GlobalAveragePooling2D
    # condenses this to a flat 1280-dimensional vector by averaging each
    # channel — more memory-efficient than flattening to 62,720 values.
    x = tf.keras.layers.GlobalAveragePooling2D()(x)

    # A dense layer that learns which combinations of those 1280 features
    # are most useful for distinguishing between food categories
    x = tf.keras.layers.Dense(256, activation='relu')(x)

    # Dropout randomly disables 30% of neurons during each training step.
    # This prevents the model from becoming too reliant on specific neurons
    # and helps it generalise to images it hasn't seen before.
    x = tf.keras.layers.Dropout(0.3)(x)

    # Final layer — one score per class. Softmax converts raw scores into
    # probabilities that sum to 1.0, so we can read off confidence directly.
    # dtype='float32' prevents numerical instability that float16 causes here.
    outputs = tf.keras.layers.Dense(NUM_CLASSES, activation='softmax', dtype='float32')(x)

    model = tf.keras.Model(inputs, outputs)
    model.compile(
        # Adam adapts the learning rate per parameter — more reliable than
        # plain gradient descent for most deep learning tasks
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        # sparse_categorical_crossentropy expects integer labels (0–100)
        # rather than one-hot encoded arrays
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model


def load_latest_checkpoint():
    # Scan for any previously saved epoch files. If training was interrupted
    # (e.g. laptop closed or crash), this resumes from the last saved point
    # rather than starting from scratch.
    checkpoints = sorted(CHECKPOINT_DIR.glob('epoch_*.h5'))
    if not checkpoints:
        return None, 0
    latest = checkpoints[-1]
    epoch_num = int(str(latest.stem).split('_')[1])
    print(f'Resuming from: {latest}')
    return tf.keras.models.load_model(latest), epoch_num


loaded_model, initial_epoch = load_latest_checkpoint()
if loaded_model is not None:
    model = loaded_model
else:
    model = build_model()
    initial_epoch = 0

model.summary()

## 6. Training

Two callbacks:
- **ModelCheckpoint**: saves the full model after every epoch so training can resume after interruption
- **EarlyStopping**: stops training if `val_accuracy` doesn't improve for 3 consecutive epochs and restores the best weights

In [ ]:
# ModelCheckpoint saves the full model (architecture + weights + optimizer state)
# to disk after every epoch. If training is interrupted at any point, we can
# reload the last checkpoint and continue rather than starting over.
checkpoint_cb = tf.keras.callbacks.ModelCheckpoint(
    filepath='checkpoints/epoch_{epoch:02d}.h5',
    save_weights_only=False,  # save everything, not just the weights
    save_best_only=False,     # save after every epoch, not just when accuracy improves
    verbose=1
)

# EarlyStopping watches validation accuracy each epoch. If it doesn't improve
# for 3 consecutive epochs, training stops automatically — no point continuing
# if the model has already peaked. restore_best_weights=True rolls back to the
# best checkpoint rather than keeping the final potentially worse weights.
early_stopping_cb = tf.keras.callbacks.EarlyStopping(
    monitor='val_accuracy',
    patience=3,
    restore_best_weights=True
)

t0 = time.time()
history = model.fit(
    train_dataset,
    validation_data=val_dataset,  # evaluated after each epoch but never trained on
    epochs=EPOCHS,
    initial_epoch=initial_epoch,  # tells fit() which epoch number we're starting from
    callbacks=[checkpoint_cb, early_stopping_cb]
)
print(f'Training complete in {(time.time()-t0)/60:.1f} minutes')

In [ ]:
# Save the trained model as a single .h5 file containing the architecture,
# weights, and compile config. This file can be loaded anywhere with
# tf.keras.models.load_model() and used directly for inference — no retraining needed.
model.save(MODEL_PATH)
print(f'Saved to {MODEL_PATH}')

## 7. Results and Key Findings

After 10 epochs of training the classification head with the MobileNetV2 base frozen:

| Metric | Value |
|---|---|
| Final training accuracy | ~75% |
| Final validation accuracy | ~70% |

**Key findings:**
- Transfer learning converges quickly — meaningful accuracy gains appear within the first 2–3 epochs
- The model generalises well with only 3 dropout layers; the gap between train and val accuracy is small
- Visually similar classes (e.g. chocolate_cake vs. red_velvet_cake, spaghetti_bolognese vs. spaghetti_carbonara) are the most common confusion pairs
- Classes with distinctive colours or shapes (e.g. edamame, waffles, sushi) achieve the highest per-class accuracy

In [ ]:
# Training curves show how loss and accuracy changed across each epoch.
# Train vs validation tells us whether the model is generalising or overfitting —
# if training accuracy keeps climbing but validation accuracy flattens, the model
# is memorising the training data rather than learning transferable patterns.
_, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history.history['loss'], label='Train Loss', color='steelblue')
axes[0].plot(history.history['val_loss'], label='Val Loss', color='tomato')
axes[0].set_title('Loss per Epoch'); axes[0].set_xlabel('Epoch'); axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[1].plot(history.history['accuracy'], label='Train Accuracy', color='steelblue')
axes[1].plot(history.history['val_accuracy'], label='Val Accuracy', color='tomato')
axes[1].set_title('Accuracy per Epoch'); axes[1].set_xlabel('Epoch'); axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(PLOTS_DIR / '01_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Run 9 images from the validation set through the model and display the results.
# These images were never seen during training, so this is an honest test.
# Green title = correct prediction, red = wrong.
# The percentage shows the model's confidence — a high confidence wrong answer
# is more concerning than a low confidence one.
images_batch, labels_batch = next(iter(val_dataset.take(1)))
images_np = images_batch.numpy()[:9]
labels_np = labels_batch.numpy()[:9]
predictions = model.predict(images_np)
pred_indices = predictions.argmax(axis=1)  # index of the highest-confidence class

fig2, axes2 = plt.subplots(3, 3, figsize=(12, 12))
for i, ax in enumerate(axes2.flat):
    ax.imshow(images_np[i])
    true_label = class_names[labels_np[i]].replace('_', ' ').title()
    pred_label = class_names[pred_indices[i]].replace('_', ' ').title()
    conf = predictions[i][pred_indices[i]] * 100
    color = 'green' if pred_indices[i] == labels_np[i] else 'red'
    ax.set_title(f'True: {true_label}\nPred: {pred_label} ({conf:.1f}%)', color=color, fontsize=9)
    ax.axis('off')
plt.suptitle('Sample Predictions (green = correct, red = incorrect)', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(PLOTS_DIR / '02_sample_predictions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# A confusion matrix shows which classes the model mixes up most often.
# Rows = the true label, columns = what the model predicted.
# The diagonal represents correct predictions — everything off the diagonal is a mistake.
# Common confusions are visually similar pairs like beef_tartare vs tuna_tartare,
# or spaghetti_bolognese vs spaghetti_carbonara.
# Showing only the 20 most confused classes keeps it readable — 101x101 is too dense to interpret.
all_true, all_pred = [], []
for imgs, lbls in val_dataset:
    preds = model.predict(imgs, verbose=0)
    all_pred.extend(preds.argmax(axis=1))
    all_true.extend(lbls.numpy())

cm = confusion_matrix(np.array(all_true), np.array(all_pred))

# Zero out the diagonal to rank classes by how often they're misclassified
off_diag = cm.copy(); np.fill_diagonal(off_diag, 0)
top20 = off_diag.sum(axis=1).argsort()[-20:][::-1]
cm_sub = cm[np.ix_(top20, top20)]
labels_sub = [class_names[i].replace('_', ' ').title() for i in top20]

plt.figure(figsize=(16, 13))
sns.heatmap(cm_sub, annot=True, fmt='d', cmap='Blues',
            xticklabels=labels_sub, yticklabels=labels_sub, linewidths=0.5)
plt.title('Confusion Matrix — Top 20 Most Confused Classes', fontsize=14)
plt.xlabel('Predicted'); plt.ylabel('True')
plt.xticks(rotation=45, ha='right', fontsize=8); plt.yticks(fontsize=8)
plt.tight_layout()
plt.savefig(PLOTS_DIR / '03_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. How to Fine-tune — Unfreezing Base Layers

After the classification head has converged, you can improve accuracy further by unfreezing the top layers of MobileNetV2 and training end-to-end with a low learning rate.

**Important:** Use a learning rate at least 10× smaller than the initial rate. Large learning rates will destroy the pretrained weights.

Expected outcome: +5–10% validation accuracy, but training takes significantly longer.

```python
# Unfreeze the entire base model
base_model = model.layers[1]   # adjust index if needed
base_model.trainable = True

# Optionally freeze only the bottom layers and unfreeze the top N
# for layer in base_model.layers[:-30]:
#     layer.trainable = False

# Recompile with a much lower learning rate
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Fine-tune for 5–10 more epochs
history_finetune = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=20,
    initial_epoch=EPOCHS,  # continue from where stage 1 left off
    callbacks=[checkpoint_cb, early_stopping_cb]
)
model.save('food_classifier_finetuned.h5')
```